# Clase 11 · Varias variables a la vez, y cómo saber si el modelo sirve

**Estadística Descriptiva e Inferencial** · Módulo 4 · Sesión 11 de 14

---

## De dónde venimos y a dónde vamos

En la Clase 10 explicamos el tiempo de resolución con **una sola** variable: la
experiencia. Pero el tiempo también depende de **qué tan complicado es el caso**.

Y aquí hay una trampa: si los analistas con más experiencia reciben los casos más
difíciles, mirar solo la experiencia puede llevarte a una conclusión **completamente
equivocada**.

Es exactamente el problema de la paradoja de Simpson (Clase 9), ahora con números.
**Hoy lo resolvemos.**

## Las dos mitades de la clase

| | Pregunta | Herramienta |
|---|---|---|
| **Primera mitad** | ¿Cómo meto varias variables en un modelo? | Regresión múltiple |
| **Segunda mitad** | ¿Cómo sé si mi modelo sirve de verdad? | Train / test |

La segunda mitad es nueva en todo el curso: hasta ahora describíamos e inferíamos.
Hoy además **predecimos**, y predecir obliga a preguntarse si el modelo funcionará con
datos que no ha visto.

## El laboratorio

| Bloque | Min | Qué haces |
|---|---|---|
| 1 | 12 | Una variable dice que no hay relación. Dos dicen que sí |
| 2 | 10 | Interpretas los coeficientes correctamente |
| 3 | 10 | Descubres que R² **siempre** sube, aunque agregues basura |
| 4 | 13 | Separas los datos en entrenamiento y prueba |
| 5 | 10 | **Ves un modelo que parece mejor y es mucho peor** |

---
## Celda 0 · Preparación

Hoy usamos dos librerías nuevas, las dos ya instaladas en Colab:

- **`statsmodels`** — para ver los coeficientes con sus p-valores
- **`scikit-learn`** — para separar en entrenamiento/prueba y medir el error

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

# ── Los datos: 60 casos de fraude resueltos ──────────────────────────────
n = 60
experiencia = np.random.default_rng(42).integers(2, 49, n).astype(float)
g = np.random.default_rng(3)
complejidad = np.clip(1 + 0.12 * experiencia + g.normal(0, 1.0, n), 1, 10).round(1)
tiempo = (12 - 0.25 * experiencia + 2.2 * complejidad + g.normal(0, 1.2, n)).round(1)

datos = pd.DataFrame({
    "experiencia": experiencia.astype(int),   # meses del analista
    "complejidad": complejidad,               # puntaje del caso, de 1 a 10
    "tiempo": tiempo,                         # dias que tardo en resolverse
})

print(f"{len(datos)} casos resueltos")
print(datos.head(6).to_string(index=False))

---
# Bloque 1 · Una variable dice una cosa, dos dicen otra  ·  12 min

Empecemos como en la Clase 10: solo con la experiencia.

### Ejercicio 1.1 — La regresión simple

`sm.OLS(y, X).fit()` ajusta la recta. Hay que añadir una columna de unos con
`sm.add_constant(X)` para que calcule el intercepto.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
y = datos["tiempo"]
X_simple = sm.add_constant(datos[["experiencia"]])

modelo_simple = sm.OLS(y, X_simple).fit()

coef_exp_simple = modelo_simple.params["experiencia"]
p_exp_simple    = modelo_simple.pvalues["experiencia"]
r2_simple       = modelo_simple.rsquared

print("REGRESION SIMPLE: tiempo ~ experiencia")
print("-" * 52)
print(f"  coeficiente de experiencia = {coef_exp_simple:+.4f}")
print(f"  p-valor                    = {p_exp_simple:.4f}")
print(f"  R2                         = {r2_simple:.4f}")
print("-" * 52)
print()
print("COMO SE LEERIA ESTE RESULTADO:")
print(f"  El coeficiente es {coef_exp_simple:+.3f}, practicamente cero.")
print(f"  El p-valor es {p_exp_simple:.2f}, muy por encima de 0.05.")
print(f"  El R2 es {r2_simple:.3f}: la experiencia explica el {100*r2_simple:.1f} % del tiempo.")
print()
print("  Conclusion aparente: 'la experiencia NO tiene relacion con el tiempo")
print("   de resolucion'. Y el calculo esta perfectamente bien hecho.")
print()
print("Guarda esa conclusion. En el proximo ejercicio se va a caer.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("coeficiente de experiencia (simple)", coef_exp_simple, -0.0058, tol=1e-3),
     check("p-valor", p_exp_simple, 0.8158, tol=1e-3),
     check("R cuadrado", r2_simple, 0.0009, tol=1e-3),
     check_bool("con este modelo, la experiencia parece irrelevante", p_exp_simple > 0.05)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Añade la complejidad del caso

Ahora metemos **las dos** variables. La sintaxis es idéntica: solo cambia la lista de
columnas.

$$tiempo = a + b_1 \cdot experiencia + b_2 \cdot complejidad$$

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
X_multiple = sm.add_constant(datos[["experiencia", "complejidad"]])

modelo_multiple = sm.OLS(y, X_multiple).fit()

coef_exp_mult  = modelo_multiple.params["experiencia"]
coef_comp_mult = modelo_multiple.params["complejidad"]
r2_multiple    = modelo_multiple.rsquared

print("REGRESION MULTIPLE: tiempo ~ experiencia + complejidad")
print("-" * 62)
print(f"  intercepto                 = {modelo_multiple.params['const']:+.4f}")
print(f"  coeficiente de experiencia = {coef_exp_mult:+.4f}   p = {modelo_multiple.pvalues['experiencia']:.2e}")
print(f"  coeficiente de complejidad = {coef_comp_mult:+.4f}   p = {modelo_multiple.pvalues['complejidad']:.2e}")
print(f"  R2                         = {r2_multiple:.4f}")
print("-" * 62)
print()
print("=" * 62)
print("COMPARA CON EL MODELO ANTERIOR:")
print(f"  coeficiente de experiencia, solo:          {coef_exp_simple:+.4f}  (p = {p_exp_simple:.2f})")
print(f"  coeficiente de experiencia, con complejidad: {coef_exp_mult:+.4f}  (p < 0.001)")
print()
print(f"  R2 paso de {r2_simple:.3f} a {r2_multiple:.3f}")
print("=" * 62)
print()
print("La conclusion se DIO VUELTA.")
print("Antes: 'la experiencia no importa'.")
print("Ahora: 'cada mes de experiencia ahorra un cuarto de dia'.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("coeficiente de experiencia (múltiple)", coef_exp_mult, -0.2229, tol=1e-3),
     check("coeficiente de complejidad", coef_comp_mult, 2.0689, tol=1e-3),
     check("R cuadrado del modelo múltiple", r2_multiple, 0.7083, tol=1e-3),
     check_bool("ahora la experiencia SÍ es significativa",
                modelo_multiple.pvalues["experiencia"] < 0.001)]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

### Ejercicio 1.3 — ¿Por qué pasó esto?

La explicación es la misma de la paradoja de Simpson: hay una variable que influye en el
resultado **y** está relacionada con la que estabas mirando.

Compruébalo: ¿están relacionadas la experiencia y la complejidad?

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
corr_exp_comp = datos["experiencia"].corr(datos["complejidad"])

print(f"correlacion entre experiencia y complejidad = {corr_exp_comp:.3f}")
print()
print("AHI ESTA LA EXPLICACION:")
print()
print("  A los analistas con mas experiencia se les asignan los casos MAS DIFICILES.")
print()
print("  Entonces, cuando miras solo la experiencia, estas mezclando dos efectos")
print("  que se cancelan:")
print("     - mas experiencia  ->  resuelve mas rapido   (efecto negativo)")
print("     - mas experiencia  ->  casos mas dificiles   (efecto positivo)")
print()
print("  El resultado neto es casi cero, y por eso el modelo simple no ve nada.")
print()
print("  Al meter la complejidad en el modelo, SEPARAS los dos efectos y")
print("  aparece el verdadero efecto de la experiencia.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(datos.experiencia, datos.complejidad, s=55, color=MAG)
ax[0].set_xlabel("experiencia (meses)"); ax[0].set_ylabel("complejidad del caso")
ax[0].set_title(f"Los expertos reciben casos difíciles (r = {corr_exp_comp:.2f})",
                color=NAVY, fontweight="bold", fontsize=11)
ax[1].scatter(datos.experiencia, datos.tiempo, s=55, color=BLUE)
ax[1].set_xlabel("experiencia (meses)"); ax[1].set_ylabel("días de resolución")
ax[1].set_title("Y por eso el tiempo no baja como debería",
                color=NAVY, fontweight="bold", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── VERIFICACIÓN 1.3 ─────────────────────────────────────────────────────
r = [check("correlación entre experiencia y complejidad", corr_exp_comp, 0.808, tol=1e-2),
     check_bool("están fuertemente relacionadas", corr_exp_comp > 0.7)]
print()
print("Esta es la respuesta al problema que dejo la Clase 9:")
print("  la regresion multiple permite CONTROLAR una variable de confusion,")
print("  metiendola en el modelo.")
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.3")

---
# Bloque 2 · Cómo se leen los coeficientes  ·  10 min

Aquí está la frase que hay que memorizar, porque es lo que distingue la regresión múltiple
de la simple:

> Cada coeficiente dice cuánto cambia `y` cuando esa variable sube una unidad
> **y todas las demás se mantienen constantes**.

Esa última parte —«manteniendo lo demás constante»— es todo el valor de la técnica.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
intercepto = modelo_multiple.params["const"]
b_exp      = modelo_multiple.params["experiencia"]
b_comp     = modelo_multiple.params["complejidad"]

print("LA ECUACION")
print("=" * 66)
print(f"  tiempo = {intercepto:.2f}  {b_exp:+.3f} x experiencia  {b_comp:+.3f} x complejidad")
print("=" * 66)
print()
print("COMO SE TRADUCE CADA NUMERO:")
print()
print(f"  {b_exp:+.3f}  experiencia")
print(f"     Por cada mes adicional de experiencia, el caso se resuelve")
print(f"     {abs(b_exp):.2f} dias mas rapido, PARA CASOS DE LA MISMA COMPLEJIDAD.")
print()
print(f"  {b_comp:+.3f}  complejidad")
print(f"     Por cada punto mas de complejidad, el caso tarda {b_comp:.2f} dias mas,")
print(f"     PARA ANALISTAS CON LA MISMA EXPERIENCIA.")
print()
print(f"  {intercepto:.2f}  intercepto")
print(f"     Lo que tardaria un analista con 0 meses en un caso de complejidad 0.")
print(f"     Aqui no significa nada real: la complejidad minima es 1.")
print()
print("LA PARTE EN MAYUSCULAS ES LA CLAVE.")
print("Sin ella, el coeficiente de experiencia era casi cero.")
print("Con ella, aparece el efecto real.")

# una prediccion concreta
e, c = 24, 5.0
pred = intercepto + b_exp * e + b_comp * c
print()
print(f"EJEMPLO DE PREDICCION:")
print(f"  Analista de {e} meses, caso de complejidad {c}:")
print(f"  {intercepto:.2f} + ({b_exp:.3f} x {e}) + ({b_comp:.3f} x {c}) = {pred:.2f} dias")

In [ ]:
# ── VERIFICACIÓN 2 ──────────────────────────────────────────────────────
pred_ejemplo = intercepto + b_exp * 24 + b_comp * 5.0
r = [check("intercepto", intercepto, 11.7783, tol=1e-3),
     check("predicción para 24 meses y complejidad 5", pred_ejemplo, 16.7728, tol=1e-2)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa el bloque 2")

---
# Bloque 3 · R² siempre sube. Siempre.  ·  10 min

Aquí hay una propiedad incómoda de R² que hay que conocer:

> **Cada variable que añades hace subir el R², aunque esa variable sea basura pura.**

Vamos a comprobarlo de la forma más brutal posible: agregando columnas de **números
aleatorios** que no tienen absolutamente nada que ver con el tiempo de resolución.

In [ ]:
# ── DEMOSTRACIÓN: creamos 20 columnas de ruido puro ──────────────────────
rng = np.random.default_rng(99)
datos_ruido = datos.copy()

for k in range(1, 21):
    datos_ruido[f"ruido{k}"] = rng.normal(0, 1, n).round(3)

print("Estas columnas son numeros al azar. No miden nada.")
print(datos_ruido[["tiempo", "ruido1", "ruido2", "ruido3"]].head(4).to_string(index=False))

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
base = ["experiencia", "complejidad"]

filas = []
for k in [0, 5, 10, 15, 20]:
    columnas = base + [f"ruido{i}" for i in range(1, k + 1)]
    m = sm.OLS(datos_ruido["tiempo"],
               sm.add_constant(datos_ruido[columnas])).fit()
    filas.append({"variables": len(columnas), "de_ruido": k,
                  "R2": round(m.rsquared, 4),
                  "R2_ajustado": round(m.rsquared_adj, 4)})

tabla_r2 = pd.DataFrame(filas)
print(tabla_r2.to_string(index=False))
print()
print("MIRA LAS DOS COLUMNAS:")
print()
print("  R2          sube SIEMPRE, de 0.708 a 0.794.")
print("              Y todas las variables que agregamos son numeros al azar.")
print()
print("  R2 ajustado sube un poco y despues BAJA, de 0.698 a 0.671.")
print("              Porque castiga por cada variable que agregas.")
print()
print("POR QUE PASA: con suficientes variables, alguna se va a parecer al")
print("azar a lo que quieres explicar. El modelo aprovecha esa coincidencia.")
print("Con 60 datos y 22 variables, la recta tiene mucho margen para acomodarse.")
print()
print("LA REGLA: nunca compares dos modelos con distinto numero de variables")
print("usando R2 a secas. Usa R2 ajustado... o mejor aun, el bloque 4.")

In [ ]:
# ── VERIFICACIÓN 3 ──────────────────────────────────────────────────────
r2_0  = tabla_r2.loc[tabla_r2.de_ruido == 0, "R2"].iloc[0]
r2_20 = tabla_r2.loc[tabla_r2.de_ruido == 20, "R2"].iloc[0]
aj_0  = tabla_r2.loc[tabla_r2.de_ruido == 0, "R2_ajustado"].iloc[0]
aj_20 = tabla_r2.loc[tabla_r2.de_ruido == 20, "R2_ajustado"].iloc[0]

r = [check("R2 con las 2 variables buenas", r2_0, 0.7083, tol=1e-3),
     check("R2 con 20 columnas de ruido", r2_20, 0.7937, tol=1e-3),
     check_bool("el R2 SUBIÓ al agregar basura", r2_20 > r2_0),
     check_bool("pero el R2 ajustado BAJÓ", aj_20 < aj_0)]
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa el bloque 3")

---
# Bloque 4 · Entrenamiento y prueba  ·  13 min

El R² ajustado ayuda, pero hay una forma mucho mejor —y más honesta— de saber si un
modelo sirve:

> **Guardas una parte de los datos, no dejas que el modelo los vea, y después le pides que
> los prediga.**

Es la idea más importante del análisis predictivo, y es de sentido común: si un modelo solo
funciona con los datos que ya conoce, no sirve para nada.

| | Para qué |
|---|---|
| **Entrenamiento** (70 %) | El modelo aprende de aquí |
| **Prueba** (30 %) | El modelo **nunca los ve** hasta el final |

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
X = datos[["experiencia", "complejidad"]]
y = datos["tiempo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"total:         {len(X)} casos")
print(f"entrenamiento: {len(X_train)} casos  ({100*len(X_train)/len(X):.0f} %)")
print(f"prueba:        {len(X_test)} casos  ({100*len(X_test)/len(X):.0f} %)")
print()
print("random_state=42 hace que la division sea siempre la misma.")
print("Sin ese parametro, cada vez que ejecutes te tocarian casos distintos")
print("y los resultados cambiarian un poco. Ponlo siempre que quieras")
print("que tu analisis sea reproducible.")

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check("casos de entrenamiento", len(X_train), 42),
     check("casos de prueba", len(X_test), 18),
     check_bool("juntos suman el total", len(X_train) + len(X_test) == len(X)),
     check_bool("no se solapan",
                len(set(X_train.index) & set(X_test.index)) == 0,
                "un caso no puede estar en los dos grupos")]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

### Ejercicio 4.2 — Entrena y evalúa

Ahora entrenamos con `scikit-learn`, que es la librería estándar para esto.

Y medimos con tres números:

| Métrica | Qué es | Se lee |
|---|---|---|
| **R²** | fracción de variación explicada | más alto es mejor |
| **RMSE** | error típico, en las unidades de y | más bajo es mejor |
| **MAE** | error absoluto promedio | más bajo es mejor |

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
modelo = LinearRegression().fit(X_train, y_train)

pred_train = modelo.predict(X_train)
pred_test  = modelo.predict(X_test)

r2_train = r2_score(y_train, pred_train)
r2_test  = r2_score(y_test, pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, pred_test))
mae_test   = mean_absolute_error(y_test, pred_test)

print("COEFICIENTES (entrenados solo con el 70 %)")
print(f"  experiencia = {modelo.coef_[0]:+.4f}")
print(f"  complejidad = {modelo.coef_[1]:+.4f}")
print(f"  intercepto  = {modelo.intercept_:.4f}")
print()
print(f"{'':16}{'R2':>10}{'RMSE':>10}")
print("-" * 36)
print(f"{'entrenamiento':16}{r2_train:10.4f}{rmse_train:10.4f}")
print(f"{'prueba':16}{r2_test:10.4f}{rmse_test:10.4f}")
print("-" * 36)
print()
print("COMO SE LEE ESTO:")
print(f"  El modelo explica el {100*r2_test:.0f} % de la variacion en casos que NUNCA VIO.")
print(f"  Su error tipico es de {rmse_test:.2f} dias.")
print()
print("  El rendimiento en prueba es parecido al de entrenamiento.")
print("  Eso significa que el modelo GENERALIZA: no memorizo, aprendio.")
print()
print(f"  MAE = {mae_test:.2f} dias: el error promedio en valor absoluto.")
print("  RMSE castiga mas los errores grandes; MAE los trata a todos igual.")

In [ ]:
# ── VERIFICACIÓN 4.2 ─────────────────────────────────────────────────────
r = [check("R2 en entrenamiento", r2_train, 0.7089, tol=1e-3),
     check("R2 en prueba", r2_test, 0.6722, tol=1e-3),
     check("RMSE en prueba", rmse_test, 1.2817, tol=1e-2),
     check_bool("el rendimiento en prueba es parecido al de entrenamiento",
                abs(r2_train - r2_test) < 0.15,
                "si la diferencia fuera enorme, habria sobreajuste")]
print()
print("Bloque 4 COMPLETO" if all(r) else "Revisa 4.2")

---
# Bloque 5 · Un modelo que parece mejor y es mucho peor  ·  10 min

**Este es el bloque más importante del día.**

Volvemos a las 20 columnas de ruido del bloque 3. Ahí vimos que hacían subir el R².

Ahora vamos a hacer la pregunta que de verdad importa: **¿mejoran las predicciones en datos
que el modelo no ha visto?**

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
idx_train = X_train.index
idx_test  = X_test.index

filas = []
for k in [0, 5, 10, 15, 20]:
    columnas = base + [f"ruido{i}" for i in range(1, k + 1)]
    Xk = datos_ruido[columnas]
    mod = LinearRegression().fit(Xk.loc[idx_train], y.loc[idx_train])

    p_tr = mod.predict(Xk.loc[idx_train])
    p_te = mod.predict(Xk.loc[idx_test])

    filas.append({
        "variables": len(columnas),
        "de_ruido": k,
        "R2_train": round(r2_score(y.loc[idx_train], p_tr), 4),
        "R2_test": round(r2_score(y.loc[idx_test], p_te), 4),
        "RMSE_test": round(np.sqrt(mean_squared_error(y.loc[idx_test], p_te)), 4),
    })

tabla_over = pd.DataFrame(filas)
print(tabla_over.to_string(index=False))
print()
print("=" * 66)
print("MIRA LAS DOS COLUMNAS DE R2, DE ARRIBA A ABAJO:")
print()
print(f"  R2 en ENTRENAMIENTO sube:  {tabla_over.R2_train.iloc[0]:.3f}  ->  {tabla_over.R2_train.iloc[-1]:.3f}")
print(f"  R2 en PRUEBA se desploma:  {tabla_over.R2_test.iloc[0]:.3f}  ->  {tabla_over.R2_test.iloc[-1]:.3f}")
print("=" * 66)
print()
print("El modelo con 22 variables parece MUCHO mejor si solo miras el")
print("entrenamiento. Y es MUCHO peor cuando le pides predecir casos nuevos.")
print()
print(f"El error tipico en prueba paso de {tabla_over.RMSE_test.iloc[0]:.2f} a {tabla_over.RMSE_test.iloc[-1]:.2f} dias.")
print()
print("ESTO SE LLAMA SOBREAJUSTE (overfitting).")
print("El modelo memorizo el ruido de los datos de entrenamiento en lugar de")
print("aprender el patron. Como el ruido es distinto en cada conjunto de datos,")
print("lo memorizado no sirve para nada nuevo.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tabla_over.variables, tabla_over.R2_train, "o-", color=BLUE, lw=2.5,
        label="R² en entrenamiento")
ax.plot(tabla_over.variables, tabla_over.R2_test, "o-", color=MAG, lw=2.5,
        label="R² en prueba (lo que importa)")
ax.set_xlabel("número de variables en el modelo")
ax.set_ylabel("R²")
ax.set_title("Más variables: parece mejor, es peor", color=NAVY, fontweight="bold")
ax.legend(frameon=False); plt.show()

In [ ]:
# ── VERIFICACIÓN 5 ──────────────────────────────────────────────────────
r2tr_0, r2tr_20 = tabla_over.R2_train.iloc[0], tabla_over.R2_train.iloc[-1]
r2te_0, r2te_20 = tabla_over.R2_test.iloc[0], tabla_over.R2_test.iloc[-1]

r = [check("R2 train con 2 variables", r2tr_0, 0.7089, tol=1e-3),
     check("R2 train con 22 variables", r2tr_20, 0.8547, tol=1e-3),
     check("R2 test con 2 variables", r2te_0, 0.6722, tol=1e-3),
     check("R2 test con 22 variables", r2te_20, 0.2067, tol=1e-3),
     check_bool("el R2 de entrenamiento SUBIÓ", r2tr_20 > r2tr_0),
     check_bool("pero el de prueba SE DERRUMBÓ", r2te_20 < r2te_0 - 0.3)]
print()
print("LA LECCION DEL DIA:")
print("  el rendimiento en los datos de entrenamiento NO dice si un modelo sirve.")
print("  Solo el rendimiento en datos que nunca vio lo dice.")
print()
print("LABORATORIO COMPLETO" if all(r) else "Revisa el bloque 5")

---
# Cierre

### Las dos ideas de hoy

**1 · Controlar variables.** Meter una variable en el modelo permite ver el efecto de otra
«manteniendo esa constante». Es la solución al problema que dejó la paradoja de Simpson en
la Clase 9.

**2 · Validar.** Un modelo solo demuestra que sirve prediciendo datos que nunca vio.
El rendimiento en entrenamiento no cuenta.

### Checklist de salida

- [ ] Sé leer un coeficiente diciendo «manteniendo lo demás constante».
- [ ] Sé que una variable omitida puede invertir una conclusión.
- [ ] Sé que R² siempre sube al agregar variables, aunque sean basura.
- [ ] Sé separar los datos en entrenamiento y prueba.
- [ ] Sé reconocer sobreajuste: bien en entrenamiento, mal en prueba.
- [ ] Reporto el error en unidades del negocio (RMSE en días, en soles).

### Los números del día

| | |
|---|---|
| Coef. experiencia, modelo simple | **−0.006** (p = 0.82) |
| Coef. experiencia, con complejidad | **−0.223** (p < 0.001) |
| Correlación experiencia-complejidad | 0.808 |
| R² con 2 variables → con 22 de ruido | 0.708 → **0.794** |
| R² ajustado, lo mismo | 0.698 → **0.671** |
| **R² en prueba, lo mismo** | 0.672 → **0.207** |

### Los tres errores que evita esta clase

**1 · Omitir una variable importante.** El coeficiente de la que sí miras queda
contaminado. Es el error de la Clase 9 en versión numérica.

**2 · Comparar modelos por R².** Siempre gana el que tiene más variables. Usa R² ajustado
o, mejor, prueba en datos nuevos.

**3 · Reportar el rendimiento en entrenamiento.** Es como calificar un examen con las
respuestas a la vista.

### Reto para la próxima clase

Toma un modelo o un análisis de tu trabajo donde se explique un número con otro.

Pregúntate: **¿qué variable falta?** ¿hay algo que influya en el resultado y que además
esté relacionado con la variable que estás usando?

Si la encuentras, métela en el modelo y mira si el coeficiente original cambia.

### Clase 12

Hoy predijimos un **número** (días). En la próxima predecimos un **sí o un no**: ¿este
cliente entrará en mora? ¿esta transacción es fraude?

Eso es la **regresión logística**, y es el modelo que está detrás de casi todo scorecard de
crédito. La validación cambia también: ya no sirve el RMSE, y entran la matriz de confusión
y el AUC.

---
*Estadística Descriptiva e Inferencial · Módulo 4 · Clase 11*